In [1]:
import pandas as pd

data = pd.read_csv('csi_data.csv')

C:\Users\yasir\AppData\Local\Temp\ipykernel_20828\1093900393.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
import numpy as np
import pywt

# Define functions for calculating wavelet amplitude and phase variances

WAVELET_NAME = 'db4'
LEVEL = 2


def calculate_wavelet_variance(csi):
    variances = []

    for i in range(129):
        amplitudes = csi[:, i+1].tolist()
        amplitudes = [float(i) for i in amplitudes]

        phases = csi[:, i+65].tolist()
        phases = [float(i) for i in phases]

        

        amplitude_coeffs = pywt.wavedec(amplitudes, WAVELET_NAME, LEVEL)
        phase_coeffs = pywt.wavedec(phases, WAVELET_NAME, LEVEL)

        amplitude_variances = [np.var(coeff) for coeff in amplitude_coeffs]
        phase_variances = [np.var(coeff) for coeff in phase_coeffs]

        # variances.append(
        # [np.sum(amplitude_variances), np.sum(phase_variances)])

        variances.append(np.sum(amplitude_variances))
        variances.append(np.sum(phase_variances))

    return variances


In [3]:
activity_zero_rows = data[data['activity'] == 0]

activity_zero_rows_count = len(activity_zero_rows)
print(activity_zero_rows_count)


78998


In [4]:
activity_one_rows = data[data['activity'] == 1]

activity_one_rows_count = len(activity_one_rows)
print(activity_one_rows_count)

35400


In [5]:
activity_two_rows = data[data['activity'] == 2]

activity_two_rows_count = len(activity_two_rows)
print(activity_two_rows_count)

31800


In [6]:
activity_three_rows = data[data['activity'] == 3]

activity_three_rows_count = len(activity_three_rows)
print(activity_three_rows_count)

24000


In [7]:
wavelet_variances = []

for i in range(activity_zero_rows_count // 20):
    csi = activity_zero_rows.iloc[i*20:i*20+20, 2:-1].values
    wavelet_variances.append(calculate_wavelet_variance(csi) + [0])

In [8]:
for i in range(activity_one_rows_count // 20):
    csi = activity_one_rows.iloc[i*20:i*20+20, 2:-1].values
    wavelet_variances.append(calculate_wavelet_variance(csi) + [1])

In [9]:
for i in range(activity_two_rows_count // 20):
    csi = activity_two_rows.iloc[i*20:i*20+20, 2:-1].values
    wavelet_variances.append(calculate_wavelet_variance(csi) + [2])

In [10]:
for i in range(activity_three_rows_count // 20):
    csi = activity_three_rows.iloc[i*20:i*20+20, 2:-1].values
    wavelet_variances.append(calculate_wavelet_variance(csi) + [3])

In [11]:
wavelet_variances_df = pd.DataFrame(wavelet_variances)

In [12]:
X = wavelet_variances_df.iloc[:, :-1]
Y = wavelet_variances_df.iloc[:, -1]

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


In [14]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

# Define hyperparameters to tune
leaf_size = list(range(1, 50))
n_neighbors = list(range(1, 30))
p = [1, 2]  # L1 (Manhattan) or L2 (Euclidean) distance

# Create a dictionary of hyperparameters
hyperparameters = {'leaf_size': leaf_size, 'n_neighbors': n_neighbors, 'p': p}

# Create a KNN object
knn = KNeighborsClassifier()

# Perform grid search
clf = GridSearchCV(knn, hyperparameters, cv=10)
clf.fit(X_train, y_train)

# Print the best hyperparameters
print('Best leaf_size:', clf.best_estimator_.get_params()['leaf_size'])
print('Best p:', clf.best_estimator_.get_params()['p'])
print('Best n_neighbors:', clf.best_estimator_.get_params()['n_neighbors'])
